[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/thebnbrkr/marv-audio/blob/main/notebooks/tests_colab.ipynb)

# marv-audio tests on Colab

1. The unit tests (a tiny random Whisper, no download).
2. Exactness checks on the real whisper-tiny. Each prints PASS or stops with an
   error. Runs on CPU on purpose: GPU matmuls can use reduced precision, which
   would blur the exactness checks.

In [ ]:
!git clone -q https://github.com/thebnbrkr/marv-audio
%cd marv-audio
!pip install -q -e ".[dev,experiments]"

In [ ]:
import subprocess, sys
r = subprocess.run([sys.executable, "-m", "pytest", "-q", "tests"], capture_output=True, text=True)
print(r.stdout[-2000:], r.stderr[-2000:])
assert r.returncode == 0, "unit tests failed"

In [ ]:
import numpy as np, torch
import marv, marv_audio
from transformers import WhisperForConditionalGeneration, WhisperProcessor

MODEL, REVISION = "openai/whisper-tiny", "169d4a4341b33bc18d8881c4b69c2e104e1cc0af"  # pinned
processor = WhisperProcessor.from_pretrained(MODEL, revision=REVISION)
model = WhisperForConditionalGeneration.from_pretrained(MODEL, revision=REVISION, use_safetensors=True).eval()
tok = processor.tokenizer
print(type(marv.detect_adapter(model)).__name__, "| decoder layers:", len(model.model.decoder.layers))

audio, _ = marv_audio.librispeech_clips(1)[0]
silence = np.zeros(16000, dtype=np.float32)
ids = marv_audio.prompt_ids(processor) + tok.encode(" Mister Quilter is the apostle of the middle classes", add_special_tokens=False)
real = marv_audio.whisper_inputs(processor, audio, ids)
quiet = marv_audio.whisper_inputs(processor, silence, ids)

def check(name, ok, detail=""):
    print(("PASS " if ok else "FAIL ") + name + (f"  ({detail})" if detail else ""))
    assert ok, name

In [ ]:
w = marv.capture_writes(model, None, real, positions=list(range(len(ids))))
check("writes rebuild the final residual", w.reconstruction_error() < 1e-4, f"{w.reconstruction_error():.1e}")
check("12 writes (4 layers x 3)", len(w.parts) == 12)

ctx = {"input_features": real["input_features"], "decoder_input_ids": real["decoder_input_ids"][:, :-1]}
d = marv.decompose_logit(model, None, ctx, ids[-1])  # score the true next token
check("decompose_logit adds up", d.check_error < 1e-3, f"{d.check_error:.1e}")
check("LayerNorm bias has its own row", "bias" in d.by_part())

s = marv_audio.listening_split(model, real)
check("listening_split adds up at every token", max(x.check_error for x in s) < 1e-3,
      f"{max(x.check_error for x in s):.1e}")

In [ ]:
from marv.trace import _first, all_components, component, replace_outputs

@torch.no_grad()
def run(inputs, patches=None, capture=False):
    cache, hooks = {}, []
    if capture:
        for c in all_components(model):
            hooks.append(component(model, *c).register_forward_hook(
                lambda _m, _i, o, c=c: cache.__setitem__(c, _first(o).detach().clone())))
    with replace_outputs(model, {c: (lambda v: (lambda x: v))(v) for c, v in (patches or {}).items()}):
        logits = model(**inputs).logits
    for h in hooks:
        h.remove()
    return (logits, cache) if capture else logits

la, ca = run(real, capture=True)
ls, cs = run(quiet, capture=True)
cross = [c for c in ca if c[1] == "cross_attn"]
diff = float((run(quiet, {c: ca[c] for c in cross}) - la).abs().max())
check("audio enters only through cross-attention", diff < 1e-4, f"max diff {diff:.1e}")

In [ ]:
feats = [(3, 10), (2, 100)]
with torch.no_grad():
    base = model(**real).logits
    with marv.suppress(model, feats):
        sup = model(**real).logits
    saved = marv.ablate(model, feats)
    abl = model(**real).logits
    marv.restore(model, saved)
    back = model(**real).logits
check("suppress == ablate", torch.allclose(sup, abl, atol=1e-5))
check("restore undoes ablate", torch.equal(back, base))

p = marv_audio.language_probs(model, processor, audio)
check("language ID sums to 1 over all languages", abs(sum(p.values()) - 1) < 1e-4)
check("English clip detected as English", next(iter(p)) == "en", f"{next(iter(p))} {max(p.values()):.2f}")
print("\nall checks passed")